# Persistence and Streaming

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [3]:
tool = TavilySearchResults(max_results=2)

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [5]:
# SqliteSaver: uses SQLite for the storage
from langgraph.checkpoint.sqlite import SqliteSaver

# Initialize a "Checkpointer". It is a DB that uato
memory = SqliteSaver.from_conn_string(":memory:") # tells SQLite to run entirely in your computer's RAM
# Fast and requires no setup
# Temporary storage.. - from_conn_string(":memory:")



In [11]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        
        # Initializes a new LangGraph. Uses AgentState to know that it needs to track a list of messages
        graph = StateGraph(AgentState)
        
        # Adding nodes
        graph.add_node("llm", self.call_openai) # for thinking
        graph.add_node("action", self.take_action) # for acting
        
        # Decision logic. After the 'llm' node, it runs **exists_action**. 
        # If it returns True, go to "action"; if False, stop (END)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        
        # After tools are used, the flow much go back to the LLM so it can read the results
        graph.add_edge("action", "llm")
        
        # entry point
        graph.set_entry_point("llm")
        
        # Turns the blueprint into a executable object and attaches the memory (SQLite)
        self.graph = graph.compile(checkpointer=checkpointer)
        
        # Creates a dict so the agent can quickly look up a tool by it's name
        self.tools = {t.name: t for t in tools}
        
        # it tells the LLM exactly what the tools are and what arguments they require
        self.model = model.bind_tools(tools)
    
    # Thinking process
    def call_openai(self, state: AgentState):
        messages = state['messages'] # gets the converstion history
        if self.system: # if system prompt
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages) # gets response
        return {'messages': [message]} # returns it
    
    # The Router
    def exists_action(self, state: AgentState):
        result = state['messages'][-1] # looks at the very last message generated by the LLM
        return len(result.tool_calls) > 0 # checks if the LLM included any "tool_calls" in it's response
    
    # The Executioner
    def take_action(self, state: AgentState):
        # extracts the specific tool requests
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [12]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [13]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [14]:
thread = {"configurable": {"thread_id": "1"}}

In [16]:
# Run the agent and watch it work in real-time.
for event in abot.graph.stream({"messages": messages}, thread): # pass the input and the thread config
    for v in event.values(): # the stream yeilds Events. "event" is a dict where the 'key' is the name of the node that just finished
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_bmfLa92f6oAIKN9KvXtqbKDz', 'function': {'arguments': '{"query":"current weather in San Francisco"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 151, 'total_tokens': 173, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_831e067d82', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-27a6597a-a517-4371-85ea-bb2fa1e937e2-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_bmfLa92f6oAIKN9KvXtqbKDz'}])]
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_bmfLa92f6oAIKN9KvXtqbKDz'}
Back t

In [17]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_kmobrb9le0QRnP9DsK8pqZv6', 'function': {'arguments': '{"query":"current weather in Los Angeles"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 1900, 'total_tokens': 1922, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_64dfa806c7', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-8b3ee304-c5a1-4a61-840d-12966639f32c-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Los Angeles'}, 'id': 'call_kmobrb9le0QRnP9DsK8pqZv6'}])]}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Los Angeles'}, 'id': 'call_kmobrb9le0QRnP9DsK8pqZv

In [18]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Currently, Los Angeles is warmer than San Francisco. Los Angeles has a temperature of 54.0°F (approximately 12.2°C), whereas San Francisco is at 51.1°F (approximately 10.6°C).', response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 3287, 'total_tokens': 3335, 'prompt_tokens_details': {'cached_tokens': 3200, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_64dfa806c7', 'finish_reason': 'stop', 'logprobs': None}, id='run-ae6bfdbe-9bbe-4b9b-bbf9-11b55410b687-0')]}


In [19]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Could you please clarify what you're comparing to determine which is warmer? Are you comparing two specific locations, types of clothing, materials, or something else? Let me know so I can provide the appropriate information.", response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 149, 'total_tokens': 192, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_f9f4fb6dbf', 'finish_reason': 'stop', 'logprobs': None}, id='run-a50b79c9-7cd0-40c9-9c1d-71dff922330a-0')]}


## Streaming tokens

In [20]:
# Asychornous input output sqlite
# to handle multiple users or stream text chunks without freezing the program
from langgraph.checkpoint.aiosqlite import AsyncSqliteSaver

memory = AsyncSqliteSaver.from_conn_string(":memory:") # RAM memory
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [21]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}} # new thread

# emits every tiny "event" happening inside the graph
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"] # filter for the "evet"
    if kind == "on_chat_model_stream": # specific event type. Triggers every time the LLM sends a new piece of text.
        content = event["data"]["chunk"].content # extracts actual text
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_GiHnpbt7P6kuX4g2n6imqDXI'}


/usr/local/lib/python3.11/site-packages/langchain_core/_api/beta_decorator.py:87: LangChainBetaWarning: This API is in beta and may change in the future.
  warn_beta(


Back to the model!
I| couldn't| find| the| current| weather| conditions| for| San| Francisco| in| the| search| results|.| You| might| want| to| check| a| weather| website| like| Weather|.com| or| Acc|u|Weather| for| the| latest| updates| on| the| current| weather| in| San| Francisco|.|